In [1]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║          BITCOIN PATTERN RECOGNITION — XGBOOST MODEL                       ║
║                    Master's Thesis — Complete Implementation                ║
╚══════════════════════════════════════════════════════════════════════════════╝

WHAT THIS FILE COVERS:
  1.  Data preparation  — synthetic BTC data + full feature engineering
  2.  Temporal split    — train / validation / test (no data leakage)
  3.  XGBoost theory    — what it is, how it works, why it suits Bitcoin
  4.  Hyperparameter tuning — manual grid search + TimeSeriesSplit CV
  5.  Classification model — predict direction (UP / DOWN)
  6.  Regression model  — predict return magnitude (%)
  7.  Feature importance — built-in gain importance + permutation importance
  8.  Threshold tuning  — optimise decision boundary for F1
  9.  Walk-forward backtest — simulate real trading evaluation
  10. Full evaluation   — accuracy, F1, ROC-AUC, MAE, RMSE, equity curve
  11. All plots saved   — for thesis figures

INSTALL:
    pip install xgboost scikit-learn optuna shap matplotlib seaborn
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, confusion_matrix,
    mean_absolute_error, mean_squared_error, r2_score,
    roc_curve, ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance
import json, joblib

# XGBoost — install: pip install xgboost
try:
    from xgboost import XGBClassifier, XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("⚠  xgboost not installed. Run: pip install xgboost")

# Optuna for Bayesian hyperparameter search
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    HAS_OPTUNA = True
except ImportError:
    HAS_OPTUNA = False
    print("⚠  optuna not installed. Using manual grid search.")

# SHAP for model explainability
try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    print("⚠  shap not installed. Run: pip install shap")

# ── output folders ──────────────────────────────────────────────────────────
for d in ["results/xgboost/plots", "results/xgboost/metrics", "models"]:
    Path(d).mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 130, "font.size": 11})


# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  STEP 1 — DATA PREPARATION & FEATURE ENGINEERING                        ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def generate_btc_data(start="2017-01-01", end="2024-12-31") -> pd.DataFrame:
    """
    Generate realistic synthetic BTC daily OHLCV data.
    Replace this with your real data:
        df = pd.read_csv('your_btc_data.csv', index_col=0, parse_dates=True)
    """
    dates = pd.date_range(start, end, freq="D")
    n     = len(dates)

    # Simulate BTC-like price with bull/bear regimes
    log_returns = []
    regime      = 1  # 1=bull, -1=bear
    for i in range(n):
        if np.random.rand() < 0.003:          # regime flip probability
            regime *= -1
        mu    = 0.002 * regime                 # bull: +0.2%/day, bear: -0.2%/day
        sigma = np.random.uniform(0.02, 0.05)  # varying volatility
        log_returns.append(np.random.normal(mu, sigma))

    price  = 1000.0 * np.exp(np.cumsum(log_returns))
    volume = np.abs(np.random.lognormal(18, 0.6, n)
                    * (1 + 2 * np.abs(np.array(log_returns))))

    df = pd.DataFrame({
        "open":   price * np.exp(np.random.normal(0, 0.003, n)),
        "high":   price * np.exp(np.abs(np.random.normal(0, 0.015, n))),
        "low":    price * np.exp(-np.abs(np.random.normal(0, 0.015, n))),
        "close":  price,
        "volume": volume,
    }, index=dates)
    df.index.name = "datetime"
    return df


def build_features(df: pd.DataFrame, horizon: int = 1) -> pd.DataFrame:
    """
    Engineer all features from raw OHLCV data.

    Features are grouped into 5 categories — each justified by the
    diagnostic steps (stationarity, ACF/PACF, ARCH, distribution):

    ① Price features   — log-return, range, body size
    ② Volume features  — log-volume, volume change, OBV
    ③ Lag features     — past returns/volume at key lags
    ④ Rolling stats    — mean, std, skew, z-score over multiple windows
    ⑤ Technical indicators — RSI, MACD, Bollinger Bands, ATR, ADX, Stochastic
    """
    df = df.copy()
    C, H, L, O, V = df.close, df.high, df.low, df.open, df.volume

    # ── ① Price features ───────────────────────────────────────────────────
    df["log_return"]    = np.log(C / C.shift(1))
    df["daily_range"]   = (H - L) / C
    df["body_size"]     = np.abs(C - O) / O
    df["upper_shadow"]  = (H - np.maximum(O, C)) / C
    df["lower_shadow"]  = (np.minimum(O, C) - L) / C
    df["gap"]           = (O - C.shift(1)) / C.shift(1)
    df["price_accel"]   = df["log_return"].diff()

    # ── ② Volume features ──────────────────────────────────────────────────
    df["log_volume"]      = np.log1p(V)
    df["volume_change"]   = V.pct_change()
    df["turnover"]        = np.log1p(C * V)

    # On-Balance Volume (OBV) — cumulative volume with direction
    obv = [0]
    for i in range(1, len(df)):
        obv.append(obv[-1] + (V.iloc[i] if C.iloc[i] > C.iloc[i-1]
                               else -V.iloc[i] if C.iloc[i] < C.iloc[i-1]
                               else 0))
    df["obv_norm"] = np.log1p(np.abs(obv)) * np.sign(obv)

    roll20_v = V.rolling(20)
    df["vol_zscore_20"] = (V - roll20_v.mean()) / (roll20_v.std() + 1e-9)

    # ── ③ Lag features ─────────────────────────────────────────────────────
    # Lags chosen from ACF/PACF analysis: 1,2,3 (short-term),
    # 5 (weekly), 7 (calendar week), 14,21,30 (medium-term)
    for lag in [1, 2, 3, 5, 7, 14, 21, 30]:
        df[f"ret_lag_{lag}"]  = df["log_return"].shift(lag)
        df[f"vol_lag_{lag}"]  = df["log_volume"].shift(lag)
        df[f"rng_lag_{lag}"]  = df["daily_range"].shift(lag)

    # ── ④ Rolling statistics ───────────────────────────────────────────────
    for w in [7, 14, 30, 60]:
        r = df["log_return"].rolling(w, min_periods=w // 2)
        df[f"ret_mean_{w}"]   = r.mean()
        df[f"ret_std_{w}"]    = r.std()
        df[f"ret_skew_{w}"]   = r.skew()
        df[f"ret_kurt_{w}"]   = r.kurt()
        df[f"ret_zscore_{w}"] = (df["log_return"] - r.mean()) / (r.std() + 1e-9)
        df[f"sharpe_{w}"]     = r.mean() / (r.std() + 1e-9)
        v = df["log_volume"].rolling(w, min_periods=w // 2)
        df[f"vol_mean_{w}"]   = v.mean()

    # ── ⑤ Technical indicators ─────────────────────────────────────────────
    def ema(s, span):
        return s.ewm(span=span, adjust=False).mean()

    # RSI (14)
    delta = C.diff()
    gain  = delta.clip(lower=0).ewm(alpha=1/14, adjust=False).mean()
    loss  = (-delta).clip(lower=0).ewm(alpha=1/14, adjust=False).mean()
    df["rsi_14"] = 100 - 100 / (1 + gain / (loss + 1e-9))
    df["rsi_centered"] = df["rsi_14"] - 50   # centre for model learning

    # MACD
    ema12 = ema(C, 12); ema26 = ema(C, 26)
    macd_line   = ema12 - ema26
    macd_signal = ema(macd_line, 9)
    df["macd_hist"]      = macd_line - macd_signal
    df["macd_hist_norm"] = df["macd_hist"] / (C + 1e-9)

    # Bollinger Bands
    sma20       = C.rolling(20).mean()
    std20       = C.rolling(20).std()
    df["bb_width"] = (2 * 2 * std20) / (sma20 + 1e-9)
    df["bb_pct"]   = (C - (sma20 - 2 * std20)) / (4 * std20 + 1e-9)

    # ATR (14) — measures true volatility
    prev_c  = C.shift(1)
    tr      = pd.concat([H-L, (H-prev_c).abs(), (L-prev_c).abs()], axis=1).max(1)
    df["atr_norm"] = ema(tr, 14) / (C + 1e-9)

    # Stochastic %K/%D
    lo14 = L.rolling(14).min(); hi14 = H.rolling(14).max()
    stoch_k         = 100 * (C - lo14) / (hi14 - lo14 + 1e-9)
    df["stoch_k"]   = stoch_k
    df["stoch_d"]   = stoch_k.rolling(3).mean()
    df["stoch_diff"]= df["stoch_k"] - df["stoch_d"]

    # Williams %R
    df["williams_r"] = -100 * (hi14 - C) / (hi14 - lo14 + 1e-9)

    # CCI
    tp  = (H + L + C) / 3
    mad = tp.rolling(20).apply(lambda x: np.abs(x - x.mean()).mean(), raw=True)
    df["cci_20"] = (tp - tp.rolling(20).mean()) / (0.015 * mad + 1e-9)

    # ADX
    up_move  = H.diff(); down_move = (-L).diff()
    plus_dm  = np.where((up_move > down_move) & (up_move > 0), up_move, 0)
    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0)
    atr14    = ema(tr, 14)
    plus_di  = 100 * ema(pd.Series(plus_dm,  index=C.index), 14) / (atr14 + 1e-9)
    minus_di = 100 * ema(pd.Series(minus_dm, index=C.index), 14) / (atr14 + 1e-9)
    dx       = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di + 1e-9)
    df["adx_14"]  = ema(dx, 14)
    df["di_diff"] = plus_di - minus_di

    # Price relative to EMAs (scale-invariant)
    for span in [7, 25, 50, 99]:
        df[f"ema_ratio_{span}"] = C / ema(C, span) - 1

    # Calendar / cyclical features
    df["dow_sin"]    = np.sin(2 * np.pi * df.index.dayofweek / 7)
    df["dow_cos"]    = np.cos(2 * np.pi * df.index.dayofweek / 7)
    df["month_sin"]  = np.sin(2 * np.pi * df.index.month / 12)
    df["month_cos"]  = np.cos(2 * np.pi * df.index.month / 12)
    df["halving_era"]= pd.cut(df.index,
        bins=[pd.Timestamp("2009-01-01"), pd.Timestamp("2016-07-09"),
              pd.Timestamp("2020-05-11"), pd.Timestamp("2024-04-20"),
              pd.Timestamp("2030-01-01")],
        labels=[1, 2, 3, 4]).astype(float)

    # ── Target variables ───────────────────────────────────────────────────
    # horizon=1 means predict TOMORROW using TODAY's features
    df["target_return"] = df["log_return"].shift(-horizon) * 100  # in %
    df["target_dir"]    = (df["target_return"] > 0).astype(int)   # 1=UP, 0=DOWN

    # Drop NaN rows created by lags and indicators
    df.dropna(inplace=True)
    return df


def temporal_split(df, train_end="2021-12-31", val_end="2022-12-31"):
    """
    Strict chronological split — NEVER random shuffle for time series.

    Train : 2017 → 2021  (learns patterns from 2 bull + 1 bear cycle)
    Val   : 2022         (2022 bear market — used for hyperparameter tuning)
    Test  : 2023 → 2024  (completely unseen — final reported numbers)
    """
    train = df.loc[:train_end]
    val   = df.loc[train_end:val_end].iloc[1:]
    test  = df.loc[val_end:].iloc[1:]

    print(f"\n{'━'*55}")
    print(f"  TEMPORAL SPLIT")
    print(f"{'━'*55}")
    print(f"  Train : {train.index[0].date()} → {train.index[-1].date()}  "
          f"({len(train):,} rows)")
    print(f"  Val   : {val.index[0].date()} → {val.index[-1].date()}  "
          f"({len(val):,} rows)")
    print(f"  Test  : {test.index[0].date()} → {test.index[-1].date()}  "
          f"({len(test):,} rows)")
    print(f"  UP%   : train={train.target_dir.mean()*100:.1f}%  "
          f"val={val.target_dir.mean()*100:.1f}%  "
          f"test={test.target_dir.mean()*100:.1f}%")
    print(f"{'━'*55}")
    return train, val, test


# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  STEP 2 — XGBOOST THEORY                                                ║
# ╚══════════════════════════════════════════════════════════════════════════╝
"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
WHAT IS XGBOOST?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
XGBoost = eXtreme Gradient Boosting
Author  : Chen & Guestrin, 2016 (one of the most cited ML papers)

Core idea: build an ensemble of weak decision trees, where each
new tree corrects the mistakes of all previous trees.

Algorithm step by step:
  1. Start: predict the mean (regression) or log-odds (classification)
  2. Compute residuals = actual - current prediction
  3. Fit a shallow decision tree to predict those residuals
  4. Add scaled tree to the ensemble: ŷ += learning_rate × tree(x)
  5. Recompute residuals, repeat for n_estimators trees

The objective function being minimised:
  Obj = Σ L(yᵢ, ŷᵢ) + Σ Ω(fₖ)
        ┗━ prediction loss   ┗━ regularisation on tree complexity
  where Ω(f) = γ·T + ½λ·‖w‖²
        T = number of leaves  (penalises complex trees)
        w = leaf weights      (penalises large weights)

KEY HYPERPARAMETERS:
  n_estimators     : how many trees to build (more = better fit, risk overfit)
  max_depth        : how deep each tree grows (deeper = more non-linear)
  learning_rate(η) : shrinkage per tree (lower = better generalisation,
                     needs more trees to compensate)
  subsample        : fraction of rows sampled per tree (adds randomness,
                     reduces variance, prevents overfitting)
  colsample_bytree : fraction of features sampled per tree (like random
                     forest — reduces correlation between trees)
  min_child_weight : minimum sum of weights in a leaf (prevents learning
                     from tiny groups, key regulariser)
  reg_alpha (L1)   : sparse weight regularisation (drives some weights to 0)
  reg_lambda (L2)  : ridge regularisation (shrinks all weights)
  gamma            : minimum loss reduction to make a split (pruning)

WHY XGBOOST FOR BITCOIN?
  ✓ Handles all 80+ features naturally — RSI, lags, rolling stats
  ✓ Non-linear: captures "RSI < 30 AND volume spike → strong UP"
  ✓ Regularisation prevents overfitting on noisy financial data
  ✓ Feature importance via gain — tells you WHAT drives Bitcoin price
  ✓ SHAP explainability — tells you WHY each prediction was made
  ✓ Fast: 500 trees on 2,000 rows takes seconds
  ✓ In literature (2019-2024): consistently top-3 for crypto prediction
"""


# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  STEP 3 — HYPERPARAMETER TUNING                                          ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def tune_xgboost(X_train, y_train, task="classification",
                 n_splits=5, n_trials=50):
    """
    Hyperparameter tuning using TimeSeriesSplit cross-validation.

    ──────────────────────────────────────────────────────────────────────
    METHOD A: Optuna (Bayesian / TPE search) — RECOMMENDED
    ──────────────────────────────────────────────────────────────────────
    Bayesian search builds a probabilistic model of which hyperparameter
    combinations are likely to be good, and focuses trials there.
    Much more efficient than grid search for 8+ hyperparameters.

    METHOD B: Manual grid search — fallback if Optuna not installed
    ──────────────────────────────────────────────────────────────────────
    Tests a predefined grid of combinations. Slower but transparent.

    WHY TIMESERIESSPLIT, NOT KFOLD?
    ──────────────────────────────────────────────────────────────────────
    TimeSeriesSplit maintains temporal order in every fold:
      Fold 1: Train[0:200]   → Val[200:250]
      Fold 2: Train[0:250]   → Val[250:300]
      Fold 3: Train[0:300]   → Val[300:350]
    Each fold's val period is always AFTER its train period.
    KFold would randomly mix past and future data — invalid for time series.
    """
    tscv    = TimeSeriesSplit(n_splits=n_splits, gap=1)
    metric  = "accuracy" if task == "classification" else "neg_mean_absolute_error"

    if HAS_OPTUNA and HAS_XGB:
        # ── Optuna Bayesian search ─────────────────────────────────────────
        def objective(trial):
            params = {
                "n_estimators":     trial.suggest_int("n_est",   100, 800),
                "max_depth":        trial.suggest_int("depth",   2,   8),
                "learning_rate":    trial.suggest_float("lr",    0.005, 0.3, log=True),
                "subsample":        trial.suggest_float("ss",    0.5,   1.0),
                "colsample_bytree": trial.suggest_float("cs",    0.4,   1.0),
                "min_child_weight": trial.suggest_int("mcw",     1,     15),
                "reg_alpha":        trial.suggest_float("l1",    1e-5,  10,  log=True),
                "reg_lambda":       trial.suggest_float("l2",    1e-5,  10,  log=True),
                "gamma":            trial.suggest_float("gamma", 0.0,   1.0),
                "random_state":     RANDOM_SEED,
                "n_jobs":           -1,
                "verbosity":        0,
            }
            scores = []
            for tr_i, val_i in tscv.split(X_train):
                X_tr, X_v = X_train.iloc[tr_i], X_train.iloc[val_i]
                y_tr, y_v = y_train.iloc[tr_i], y_train.iloc[val_i]
                if task == "classification":
                    m = XGBClassifier(**params, eval_metric="logloss",
                                      use_label_encoder=False)
                    m.fit(X_tr, y_tr, verbose=False)
                    scores.append(accuracy_score(y_v, m.predict(X_v)))
                else:
                    m = XGBRegressor(**params, eval_metric="rmse")
                    m.fit(X_tr, y_tr, verbose=False)
                    scores.append(-mean_absolute_error(y_v, m.predict(X_v)))
            return np.mean(scores)

        study = optuna.create_study(
            direction="maximize",
            sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED)
        )
        study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

        # Map short keys back to full names
        bp = study.best_params
        best = {
            "n_estimators":     bp.get("n_est",   300),
            "max_depth":        bp.get("depth",   4),
            "learning_rate":    bp.get("lr",      0.05),
            "subsample":        bp.get("ss",      0.8),
            "colsample_bytree": bp.get("cs",      0.8),
            "min_child_weight": bp.get("mcw",     3),
            "reg_alpha":        bp.get("l1",      0.1),
            "reg_lambda":       bp.get("l2",      1.0),
            "gamma":            bp.get("gamma",   0.0),
        }
        best_score = study.best_value
        method = "Optuna (Bayesian TPE)"

    else:
        # ── Manual grid search fallback ────────────────────────────────────
        print("  Running manual grid search (Optuna/XGBoost not available)…")
        param_grid = [
            {"n_estimators": ne, "max_depth": md,
             "learning_rate": lr, "subsample": ss,
             "colsample_bytree": cs, "min_child_weight": 3,
             "reg_alpha": 0.1, "reg_lambda": 1.0}
            for ne in [200, 400]
            for md in [3, 5]
            for lr in [0.05, 0.1]
            for ss in [0.8]
            for cs in [0.8]
        ]
        best, best_score = param_grid[0], -np.inf
        for params in param_grid:
            fold_scores = []
            for tr_i, val_i in tscv.split(X_train):
                X_tr, X_v = X_train.iloc[tr_i], X_train.iloc[val_i]
                y_tr, y_v = y_train.iloc[tr_i], y_train.iloc[val_i]
                # Simulate model behaviour with sklearn DecisionTree as proxy
                # Replace with XGBClassifier when installed
                from sklearn.ensemble import GradientBoostingClassifier
                m = GradientBoostingClassifier(
                    n_estimators=params["n_estimators"],
                    max_depth=params["max_depth"],
                    learning_rate=params["learning_rate"],
                    subsample=params["subsample"],
                    random_state=RANDOM_SEED
                )
                m.fit(X_tr, y_tr)
                fold_scores.append(accuracy_score(y_v, m.predict(X_v)))
            score = np.mean(fold_scores)
            if score > best_score:
                best_score, best = score, params
        method = "Manual grid search (sklearn GBM proxy)"

    print(f"\n  Tuning method    : {method}")
    print(f"  Best CV score    : {best_score*100:.2f}%")
    print(f"  Best parameters  :")
    for k, v in best.items():
        print(f"    {k:22s}: {v}")

    with open(f"results/xgboost/metrics/best_params_{task}.json", "w") as f:
        json.dump({"params": best, "cv_score": best_score}, f, indent=2)

    return best


# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  STEP 4 — XGBOOST CLASSIFICATION MODEL (direction: UP / DOWN)           ║
# ╚══════════════════════════════════════════════════════════════════════════╝

class XGBoostClassifierBTC:
    """
    Binary classifier: predicts whether next-day return > 0 (UP) or ≤ 0 (DOWN).

    Output: probability of UP (class=1)
    Default threshold: 0.5 (tuned in tune_threshold())
    """

    def __init__(self):
        self.model      = None
        self.scaler     = StandardScaler()
        self.threshold  = 0.50
        self.features   = None
        self.params     = {}

    def fit(self, X_train, y_train, X_val, y_val, params=None):
        """
        Train the classifier with early stopping on validation loss.

        Early stopping:
          Monitors val logloss every round.
          Stops when no improvement for 50 rounds.
          Uses best iteration weights — prevents overfitting automatically.
          This means n_estimators is an UPPER BOUND, not a fixed count.
        """
        self.features = list(X_train.columns)

        # Scale features — fit on TRAIN only
        X_tr_sc = self.scaler.fit_transform(X_train)
        X_v_sc  = self.scaler.transform(X_val)

        if HAS_XGB:
            p = params or {
                "n_estimators": 500, "max_depth": 4,
                "learning_rate": 0.05, "subsample": 0.8,
                "colsample_bytree": 0.8, "min_child_weight": 3,
                "reg_alpha": 0.1, "reg_lambda": 1.0, "gamma": 0.0,
            }
            self.params = p
            self.model  = XGBClassifier(
                **p,
                eval_metric="logloss",
                early_stopping_rounds=50,
                random_state=RANDOM_SEED,
                n_jobs=-1,
                verbosity=0,
            )
            self.model.fit(
                X_tr_sc, y_train,
                eval_set=[(X_v_sc, y_val)],
                verbose=False
            )
            best_iter = self.model.best_iteration
            print(f"\n  XGBoost Classifier trained")
            print(f"  Best iteration (early stopping): {best_iter}")
        else:
            # Fallback: sklearn GradientBoostingClassifier
            from sklearn.ensemble import GradientBoostingClassifier
            print("\n  Using sklearn GradientBoostingClassifier (XGBoost proxy)")
            self.model = GradientBoostingClassifier(
                n_estimators=300, max_depth=4, learning_rate=0.05,
                subsample=0.8, random_state=RANDOM_SEED
            )
            self.model.fit(X_tr_sc, y_train)
        return self

    def predict_proba(self, X):
        """Return probability of UP (class=1)."""
        X_sc = self.scaler.transform(X)
        if HAS_XGB:
            return self.model.predict_proba(X_sc)[:, 1]
        else:
            return self.model.predict_proba(X_sc)[:, 1]

    def predict(self, X):
        """Return binary predictions using current threshold."""
        return (self.predict_proba(X) >= self.threshold).astype(int)

    def tune_threshold(self, X_val, y_val):
        """
        Find the decision threshold that maximises F1-score on validation set.

        Why not use 0.5?
          Bitcoin has more UP days than DOWN days (~55% vs 45%).
          With threshold=0.5, the model is slightly biased toward predicting UP.
          Optimising threshold for F1 balances precision and recall, ensuring
          the model is not just predicting UP for every single day.

        Threshold range: 0.30 to 0.70 in steps of 0.01
        """
        proba     = self.predict_proba(X_val)
        best_f1, best_thr = 0.0, 0.5

        for thr in np.arange(0.30, 0.70, 0.01):
            pred = (proba >= thr).astype(int)
            f1   = f1_score(y_val, pred, zero_division=0)
            if f1 > best_f1:
                best_f1, best_thr = f1, thr

        self.threshold = best_thr
        print(f"  Optimal threshold : {best_thr:.2f}  "
              f"(val F1 = {best_f1:.4f})")
        return best_thr

    def feature_importance(self, top_n=25,
                           save_path="results/xgboost/plots/feature_importance.png"):
        """
        Plot feature importance using XGBoost gain metric.

        Gain = average improvement in loss brought by a feature across
               all trees and splits where that feature is used.
        A high-gain feature makes splits that significantly reduce loss.
        This is more meaningful than 'frequency' (how often used) because
        a feature used rarely but with high gain is still important.
        """
        if not HAS_XGB or self.model is None:
            print("  Feature importance requires XGBoost.")
            return

        importances = pd.Series(
            self.model.feature_importances_,
            index=self.features
        ).sort_values(ascending=False).head(top_n)

        fig, ax = plt.subplots(figsize=(10, 8))
        colors  = plt.cm.Blues(np.linspace(0.4, 0.9, top_n))[::-1]
        importances.sort_values().plot(
            kind="barh", ax=ax, color=colors, edgecolor="white"
        )
        ax.set_title(f"XGBoost Feature Importance — Top {top_n} (Gain)",
                     fontsize=13, fontweight="bold")
        ax.set_xlabel("Importance (Gain)")
        ax.axvline(importances.mean(), color="red", linestyle="--",
                   linewidth=0.8, label=f"Mean = {importances.mean():.4f}")
        ax.legend()
        plt.tight_layout()
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.close()
        print(f"  Saved: {save_path}")

        # Save to CSV for thesis table
        importances.reset_index().rename(
            columns={"index": "feature", 0: "importance"}
        ).to_csv("results/xgboost/metrics/feature_importance.csv", index=False)
        return importances

    def save(self, path="models/xgb_classifier"):
        joblib.dump(self, path + ".pkl")
        print(f"  Model saved: {path}.pkl")


# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  STEP 5 — XGBOOST REGRESSION MODEL (magnitude: return %)                ║
# ╚══════════════════════════════════════════════════════════════════════════╝

class XGBoostRegressorBTC:
    """
    Regression model: predicts exact next-day return in %.

    Directional Accuracy (DA) = % of times predicted sign == actual sign.
    This bridges regression output and classification performance.
    """

    def __init__(self):
        self.model  = None
        self.scaler = StandardScaler()

    def fit(self, X_train, y_train, X_val, y_val, params=None):
        X_tr_sc = self.scaler.fit_transform(X_train)
        X_v_sc  = self.scaler.transform(X_val)

        if HAS_XGB:
            p = params or {
                "n_estimators": 500, "max_depth": 4,
                "learning_rate": 0.05, "subsample": 0.8,
                "colsample_bytree": 0.8, "min_child_weight": 3,
                "reg_alpha": 0.1, "reg_lambda": 1.0,
            }
            self.model = XGBRegressor(
                **p, eval_metric="rmse",
                early_stopping_rounds=50,
                random_state=RANDOM_SEED, n_jobs=-1, verbosity=0
            )
            self.model.fit(X_tr_sc, y_train,
                           eval_set=[(X_v_sc, y_val)], verbose=False)
        else:
            from sklearn.ensemble import GradientBoostingRegressor
            self.model = GradientBoostingRegressor(
                n_estimators=300, max_depth=4, learning_rate=0.05,
                subsample=0.8, random_state=RANDOM_SEED
            )
            self.model.fit(X_tr_sc, y_train)
        return self

    def predict(self, X):
        return self.model.predict(self.scaler.transform(X))

    def save(self, path="models/xgb_regressor"):
        joblib.dump(self, path + ".pkl")
        print(f"  Model saved: {path}.pkl")


# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  STEP 6 — WALK-FORWARD BACKTEST                                          ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def walk_forward_backtest(df, feature_cols,
                          initial_train_end="2021-12-31",
                          retrain_every=90):
    """
    Walk-forward backtesting — the correct way to evaluate ML models
    on time series data.

    ──────────────────────────────────────────────────────────────────────
    WHAT IS WALK-FORWARD BACKTESTING?
    ──────────────────────────────────────────────────────────────────────
    Instead of training once and predicting everything, we:
      1. Train on all data up to date T
      2. Predict day T+1
      3. Add day T+1 to training data
      4. Retrain every 90 days (weekly retraining is too slow)
      5. Repeat until end of dataset

    This simulates exactly what would happen in live trading:
      - The model only ever uses past data
      - It is retrained periodically as new data arrives
      - No future data ever leaks into training

    ──────────────────────────────────────────────────────────────────────
    WHY NOT JUST TRAIN ONCE AND PREDICT?
    ──────────────────────────────────────────────────────────────────────
    A model trained up to 2021 may perform well on 2022 but poorly
    on 2024 because the market regime changed. Walk-forward testing
    adapts to regime changes — it is far more realistic and defensible
    in a thesis.
    """
    from sklearn.ensemble import GradientBoostingClassifier

    train_mask = df.index <= initial_train_end
    results    = []
    scaler     = StandardScaler()
    model      = None
    step       = 0

    test_indices = df.index[train_mask == False]
    print(f"\n  Walk-forward backtest: {len(test_indices)} out-of-sample steps")
    print(f"  Retraining every {retrain_every} steps")

    for i, dt in enumerate(test_indices):
        # Retrain?
        if model is None or i % retrain_every == 0:
            train_mask_now = df.index < dt
            X_tr = df.loc[train_mask_now, feature_cols]
            y_tr = df.loc[train_mask_now, "target_dir"]
            scaler.fit(X_tr)
            X_tr_sc = scaler.transform(X_tr)

            if HAS_XGB:
                model = XGBClassifier(
                    n_estimators=300, max_depth=4, learning_rate=0.05,
                    subsample=0.8, colsample_bytree=0.8,
                    eval_metric="logloss", random_state=RANDOM_SEED,
                    n_jobs=-1, verbosity=0
                )
            else:
                model = GradientBoostingClassifier(
                    n_estimators=200, max_depth=4, learning_rate=0.05,
                    subsample=0.8, random_state=RANDOM_SEED
                )
            model.fit(X_tr_sc, y_tr)
            step += 1

        # Predict
        row     = df.loc[[dt], feature_cols]
        X_sc    = scaler.transform(row)
        prob_up = model.predict_proba(X_sc)[0, 1]
        pred    = int(prob_up >= 0.5)
        actual  = int(df.loc[dt, "target_dir"])
        ret     = float(df.loc[dt, "target_return"])

        results.append({
            "datetime": dt, "actual": actual,
            "predicted": pred, "prob_up": prob_up, "return_pct": ret
        })

    results_df = pd.DataFrame(results).set_index("datetime")
    acc = accuracy_score(results_df.actual, results_df.predicted)
    print(f"  Walk-forward directional accuracy: {acc*100:.2f}%")
    print(f"  Total retrains performed: {step}")
    return results_df


# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  STEP 7 — EVALUATION & PLOTS                                             ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def evaluate_classifier(y_true, y_pred, y_prob,
                        label="XGBoost", split="Test"):
    """
    Full classification evaluation for thesis results chapter.
    Returns metrics dict and prints formatted table.
    """
    metrics = {
        "Accuracy":  accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall":    recall_score(y_true, y_pred, zero_division=0),
        "F1-Score":  f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC":   roc_auc_score(y_true, y_prob),
        "MCC":       matthews_corrcoef(y_true, y_pred),
    }

    print(f"\n  {'━'*45}")
    print(f"  {label} — {split} Set Classification Results")
    print(f"  {'━'*45}")
    for k, v in metrics.items():
        bar   = "█" * int(v * 20)
        stars = ("★★★" if v > 0.65 else "★★" if v > 0.55 else "★")
        print(f"  {k:12s} : {v:.4f}  {bar:<20s} {stars}")
    print(f"  {'━'*45}")

    baseline = y_true.mean()
    print(f"  Always-UP baseline accuracy: {baseline:.4f}")
    print(f"  Model vs baseline: "
          f"{'+' if metrics['Accuracy'] > baseline else ''}"
          f"{(metrics['Accuracy'] - baseline)*100:.2f}%")
    return metrics


def evaluate_regressor(y_true, y_pred, label="XGBoost", split="Test"):
    """Full regression evaluation."""
    da = (np.sign(y_pred) == np.sign(y_true)).mean()
    metrics = {
        "MAE":                  mean_absolute_error(y_true, y_pred),
        "RMSE":                 np.sqrt(mean_squared_error(y_true, y_pred)),
        "R²":                   r2_score(y_true, y_pred),
        "Directional Accuracy": da,
    }
    print(f"\n  {label} — {split} Regression Results")
    print(f"  {'━'*40}")
    for k, v in metrics.items():
        print(f"  {k:22s} : {v:.4f}")
    return metrics


def plot_xgboost_results(y_true_dir, y_pred_dir, y_prob,
                         y_true_ret, y_pred_ret,
                         test_index,
                         save_dir="results/xgboost/plots"):
    """
    Generate all thesis-quality plots for XGBoost.
    6 panels: confusion matrix, ROC curve, probability distribution,
              return scatter, equity curve, rolling accuracy.
    """
    fig = plt.figure(figsize=(18, 14))
    gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.40, wspace=0.35)

    # ── Panel 1: Confusion Matrix ──────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0, 0])
    cm  = confusion_matrix(y_true_dir, y_pred_dir)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Pred DOWN", "Pred UP"],
                yticklabels=["Actual DOWN", "Actual UP"],
                ax=ax1, cbar=False, linewidths=0.5)
    acc = accuracy_score(y_true_dir, y_pred_dir)
    ax1.set_title(f"Confusion Matrix\nAccuracy = {acc*100:.1f}%",
                  fontsize=11, fontweight="bold")

    # ── Panel 2: ROC Curve ────────────────────────────────────────────────
    ax2  = fig.add_subplot(gs[0, 1])
    fpr, tpr, _ = roc_curve(y_true_dir, y_prob)
    auc  = roc_auc_score(y_true_dir, y_prob)
    ax2.plot(fpr, tpr, color="#1565C0", lw=2, label=f"AUC = {auc:.4f}")
    ax2.plot([0,1],[0,1], "k--", lw=1, label="Random (AUC=0.5)")
    ax2.fill_between(fpr, tpr, alpha=0.1, color="#1565C0")
    ax2.set_xlabel("False Positive Rate"); ax2.set_ylabel("True Positive Rate")
    ax2.set_title("ROC Curve", fontsize=11, fontweight="bold")
    ax2.legend(fontsize=9)

    # ── Panel 3: Predicted Probability Distribution ───────────────────────
    ax3 = fig.add_subplot(gs[0, 2])
    ax3.hist(y_prob[y_true_dir == 0], bins=40, alpha=0.6,
             color="#EF5350", label="Actual DOWN", density=True)
    ax3.hist(y_prob[y_true_dir == 1], bins=40, alpha=0.6,
             color="#26A69A", label="Actual UP",   density=True)
    ax3.axvline(0.5, color="black", linestyle="--", lw=1, label="Threshold 0.5")
    ax3.set_xlabel("P(UP)"); ax3.set_ylabel("Density")
    ax3.set_title("Predicted Probability Distribution",
                  fontsize=11, fontweight="bold")
    ax3.legend(fontsize=9)

    # ── Panel 4: Actual vs Predicted Return Scatter ───────────────────────
    ax4 = fig.add_subplot(gs[1, 0])
    ax4.scatter(y_true_ret, y_pred_ret, alpha=0.3, s=10, color="#1565C0")
    lims = [min(y_true_ret.min(), y_pred_ret.min()),
            max(y_true_ret.max(), y_pred_ret.max())]
    ax4.plot(lims, lims, "r--", lw=1, label="Perfect prediction")
    ax4.axhline(0, color="gray", lw=0.5); ax4.axvline(0, color="gray", lw=0.5)
    ax4.set_xlabel("Actual Return %"); ax4.set_ylabel("Predicted Return %")
    r2 = r2_score(y_true_ret, y_pred_ret)
    ax4.set_title(f"Actual vs Predicted Return\nR² = {r2:.4f}",
                  fontsize=11, fontweight="bold")
    ax4.legend(fontsize=9)

    # ── Panel 5: Equity Curve ─────────────────────────────────────────────
    ax5   = fig.add_subplot(gs[1, 1:])
    ret   = pd.Series(y_true_ret / 100, index=test_index)
    sig   = pd.Series(y_pred_dir, index=test_index).map({1: 1, 0: -1})

    cum_bh       = (1 + ret).cumprod()
    cum_longshort= (1 + ret * sig).cumprod()
    cum_longonly = (1 + ret * (sig.clip(lower=0))).cumprod()

    ax5.plot(cum_bh.index,        cum_bh,        color="gray",    lw=1.2,
             label="Buy & Hold",   alpha=0.8)
    ax5.plot(cum_longshort.index, cum_longshort, color="#1565C0", lw=1.5,
             label="XGBoost Long/Short")
    ax5.plot(cum_longonly.index,  cum_longonly,  color="#2E7D32", lw=1.5,
             label="XGBoost Long-only")
    ax5.axhline(1.0, color="black", lw=0.5, linestyle="--")
    ax5.set_title("Equity Curve — XGBoost Strategy vs Buy & Hold",
                  fontsize=11, fontweight="bold")
    ax5.set_ylabel("Cumulative Return")
    ax5.legend(fontsize=9)
    ax5.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda x, _: f"{x:.1f}x"))

    # ── Panel 6: Rolling 30-day Directional Accuracy ─────────────────────
    ax6      = fig.add_subplot(gs[2, :])
    correct  = pd.Series(
        (np.array(y_pred_dir) == np.array(y_true_dir)).astype(float),
        index=test_index
    )
    roll_acc = correct.rolling(30).mean() * 100
    ax6.plot(roll_acc.index, roll_acc, color="#1565C0", lw=0.9)
    ax6.axhline(50, color="red", linestyle="--", lw=0.8,
                label="50% (random baseline)")
    ax6.axhline(correct.mean()*100, color="green", linestyle="-.", lw=0.8,
                label=f"Overall mean = {correct.mean()*100:.1f}%")
    ax6.fill_between(roll_acc.index, 50, roll_acc,
                     where=roll_acc > 50, alpha=0.15, color="green")
    ax6.fill_between(roll_acc.index, 50, roll_acc,
                     where=roll_acc < 50, alpha=0.15, color="red")
    ax6.set_title("30-Day Rolling Directional Accuracy",
                  fontsize=11, fontweight="bold")
    ax6.set_ylabel("Accuracy %")
    ax6.legend(fontsize=9)

    plt.suptitle("XGBoost Model — Complete Evaluation Dashboard",
                 fontsize=14, fontweight="bold", y=1.01)

    path = f"{save_dir}/xgboost_evaluation_dashboard.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"\n  Saved: {path}")


# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  MAIN — RUN EVERYTHING                                                   ║
# ╚══════════════════════════════════════════════════════════════════════════╝

if __name__ == "__main__":

    print("""
╔══════════════════════════════════════════════════════════════════════════╗
║              XGBOOST MODEL — BITCOIN PATTERN RECOGNITION                ║
╚══════════════════════════════════════════════════════════════════════════╝
""")
    # ── 1. Data ──────────────────────────────────────────────────────────
    print("━"*55)
    print("STEP 1: Data Preparation & Feature Engineering")
    print("━"*55)
    df_raw  = generate_btc_data("2017-01-01", "2024-12-31")
    df_feat = build_features(df_raw, horizon=1)
    print(f"  Feature matrix : {df_feat.shape[0]:,} rows × "
          f"{df_feat.shape[1]} columns")

    # Define feature columns (exclude raw OHLCV and targets)
    exclude = {"open","high","low","close","volume",
               "target_dir","target_return","log_return"}
    feature_cols = [c for c in df_feat.columns if c not in exclude]
    print(f"  Features used  : {len(feature_cols)}")

    # ── 2. Split ─────────────────────────────────────────────────────────
    print("\n" + "━"*55)
    print("STEP 2: Temporal Train / Val / Test Split")
    print("━"*55)
    train, val, test = temporal_split(df_feat)

    X_train = train[feature_cols];  y_train_dir = train["target_dir"]
    X_val   = val[feature_cols];    y_val_dir   = val["target_dir"]
    X_test  = test[feature_cols];   y_test_dir  = test["target_dir"]
    y_train_ret = train["target_return"]
    y_val_ret   = val["target_return"]
    y_test_ret  = test["target_return"]

    # ── 3. Hyperparameter tuning ─────────────────────────────────────────
    print("\n" + "━"*55)
    print("STEP 3: Hyperparameter Tuning (TimeSeriesSplit CV)")
    print("━"*55)
    best_clf_params = tune_xgboost(X_train, y_train_dir,
                                    task="classification",
                                    n_splits=5, n_trials=40)
    best_reg_params = tune_xgboost(X_train, y_train_ret,
                                    task="regression",
                                    n_splits=5, n_trials=40)

    # ── 4. Classification model ──────────────────────────────────────────
    print("\n" + "━"*55)
    print("STEP 4: XGBoost Classification Model (Direction)")
    print("━"*55)
    clf = XGBoostClassifierBTC()
    clf.fit(X_train, y_train_dir, X_val, y_val_dir, params=best_clf_params)
    clf.tune_threshold(X_val, y_val_dir)
    clf.feature_importance(top_n=25)
    clf.save()

    y_pred_dir = clf.predict(X_test)
    y_prob     = clf.predict_proba(X_test)
    clf_metrics = evaluate_classifier(
        y_test_dir.values, y_pred_dir, y_prob,
        label="XGBoost", split="Test"
    )

    # ── 5. Regression model ──────────────────────────────────────────────
    print("\n" + "━"*55)
    print("STEP 5: XGBoost Regression Model (Return %)")
    print("━"*55)
    reg = XGBoostRegressorBTC()
    reg.fit(X_train, y_train_ret, X_val, y_val_ret, params=best_reg_params)
    reg.save()

    y_pred_ret = reg.predict(X_test)
    reg_metrics = evaluate_regressor(
        y_test_ret.values, y_pred_ret,
        label="XGBoost", split="Test"
    )

    # ── 6. Walk-forward backtest ─────────────────────────────────────────
    print("\n" + "━"*55)
    print("STEP 6: Walk-Forward Backtest")
    print("━"*55)
    wf_results = walk_forward_backtest(df_feat, feature_cols,
                                        initial_train_end="2021-12-31",
                                        retrain_every=90)

    # ── 7. Plots ─────────────────────────────────────────────────────────
    print("\n" + "━"*55)
    print("STEP 7: Generating Evaluation Plots")
    print("━"*55)
    plot_xgboost_results(
        y_test_dir.values, y_pred_dir, y_prob,
        y_test_ret.values, y_pred_ret,
        test.index
    )

    # ── 8. Save all metrics ──────────────────────────────────────────────
    all_metrics = {
        "classification": {k: float(v) for k, v in clf_metrics.items()},
        "regression":     {k: float(v) for k, v in reg_metrics.items()},
        "walk_forward_accuracy": float(
            accuracy_score(wf_results.actual, wf_results.predicted)
        ),
    }
    with open("results/xgboost/metrics/final_metrics.json", "w") as f:
        json.dump(all_metrics, f, indent=2)

    print(f"""
╔══════════════════════════════════════════════════════════════════════════╗
║  XGBOOST COMPLETE                                                        ║
║  Classification accuracy : {clf_metrics['Accuracy']*100:5.2f}%                            ║
║  ROC-AUC                 : {clf_metrics['ROC-AUC']:.4f}                              ║
║  Regression MAE          : {reg_metrics['MAE']:.4f}%                              ║
║  Directional accuracy    : {reg_metrics['Directional Accuracy']*100:5.2f}%                            ║
║                                                                          ║
║  All plots → results/xgboost/plots/                                      ║
║  All metrics → results/xgboost/metrics/                                  ║
║  Next → run lstm_model.py                                                ║
╚══════════════════════════════════════════════════════════════════════════╝
""")

⚠  optuna not installed. Using manual grid search.
⚠  shap not installed. Run: pip install shap

╔══════════════════════════════════════════════════════════════════════════╗
║              XGBOOST MODEL — BITCOIN PATTERN RECOGNITION                ║
╚══════════════════════════════════════════════════════════════════════════╝

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
STEP 1: Data Preparation & Feature Engineering
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Feature matrix : 2,890 rows × 94 columns
  Features used  : 86

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
STEP 2: Temporal Train / Val / Test Split
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  TEMPORAL SPLIT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Train : 2017-02-01 → 2021-12-31  (1,795 rows)
  Val   : 2022-01-01 → 2022-12-31  (365 rows)
  Test  : 2023-01-01 → 2024-12-30  (730 rows)
  UP%   : train=54.3%  

ValueError: Unknown label type: continuous. Maybe you are trying to fit a classifier, which expects discrete classes on a regression target with continuous values.